In [45]:
import pandas as pd
import numpy as np

In [46]:
df= pd.read_excel('marketing_campaign_dataset-cleaned.xlsx')

In [47]:
df.head(10)

,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion %,Acquisition_Cost,ROI %,Location,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,TechCorp,Display,Women 25-34,30 days,Google Ads,5.40,14215,-12.4,Miami,4904,252755,5,Foodies,2025-05-23
1,2,TechCorp,Social Media,Men 18-24,45 days,Facebook,2.49,34240,-26.1,New York,12836,679362,8,Fashionistas,2025-12-18
2,3,TechCorp,Email,Men 18-24,15 days,Email,5.01,8342,52.6,Miami,4470,128024,6,Fashionistas,2025-07-01
3,4,NexGen Systems,Influencer,Men 25-34,15 days,YouTube,1.21,5478,-7.3,Los Angeles,3960,464236,8,Fashionistas,2025-06-13
4,5,Innovate Industries,Social Media,Women 25-34,60 days,Instagram,1.38,22373,25.0,New York,9354,744063,6,Tech Enthusiasts,2025-10-31
5,6,NexGen Systems,Search,Women 25-34,30 days,Website,6.94,9086,489.9,New York,6485,146316,6,Outdoor Adventurers,2025-02-17
6,7,Innovate Industries,Social Media,Men 25-34,30 days,Instagram,1.67,4209,-38.5,New York,1971,179401,7,Fashionistas,2025-02-27
7,8,NexGen Systems,Display,Women 25-34,15 days,Facebook,0.97,3003,-19.0,New York,1549,374616,2,Tech Enthusiasts,2025-06-26
8,9,Innovate Industries,Influencer,Women 35-44,15 days,Instagram,1.33,7749,-59.5,New York,2782,236910,6,Health & Wellness,2025-09-19
9,10,Alpha Innovations,Display,Women 25-34,45 days,Website,3.70,2635,-13.1,New York,1190,85740,1,Health & Wellness,2025-04-09


In [48]:
print(df["ROI %"].min())
print(df["ROI %"].max())

-100.0
1439.9


In [49]:
# 1. Structure: expected columns and data types
expected_cols = ["Campaign_ID","Company", "Campaign_Type", "Target_Audience","Duration", "Channel_Used",
                 "Conversion %", "Acquisition_Cost", "ROI %", "Location",
                 "Clicks","Impressions", "Engagement_Score", "Customer_Segment", "Date"]
assert list(df.columns) == expected_cols, "Column names/order differ"
assert pd.api.types.is_datetime64_any_dtype(df["Date"])
assert pd.api.types.is_numeric_dtype(df["Acquisition_Cost"])

# 2. Completeness: no missing values
assert df.isna().sum().sum() == 0

# 3. Uniqueness: no duplicate rows or IDs
assert not df.duplicated().any()
assert df["Campaign_ID"].is_unique

# 4. Range checks (numeric limits seen in your data)
assert df["Conversion %"].between(0, 100).all()            # a rate can't be below 0 or above 100%
assert (df["Acquisition_Cost"] > 0).all()                # cost must be positive
assert (df["Clicks"] >= 0).all()
assert (df["Impressions"] >= 0).all()
assert df["Engagement_Score"].between(1, 10).all()       # 1-10 scale

# 5. Allowed values (categorical columns)
allowed = {
    "Campaign_Type": {"Display", "Email", "Influencer", "Search", "Social Media"},
    "Target_Audience": {"All Ages", "Men 18-24", "Men 25-34", "Women 25-34", "Women 35-44"},
    "Channel_Used": {"Email", "Facebook", "Google Ads", "Instagram", "Website", "YouTube"},
    "Location": {"Chicago", "Houston", "Los Angeles", "Miami", "New York"},
    "Customer_Segment": {"Fashionistas", "Foodies", "Health & Wellness",
                         "Outdoor Adventurers", "Tech Enthusiasts"},
    "Duration": {"15 days", "30 days", "45 days", "60 days"},
    "Company":{"TechCorp", "NexGen Systems", "Innovate Industries", "Alpha Innovations", "DataTech Solutions"},
}
for col, valid in allowed.items():
    bad = set(df[col].unique()) - valid
    assert not bad, f"{col} has unexpected values: {bad}"

# 6. Date checks
assert df["Date"].between("2025-01-01", "2025-12-31").all()   # within the expected year
                    # optional: dates ordered by ID

# 7. Text hygiene: no stray spaces or inconsistent casing
for col in allowed:
    assert (df[col] == df[col].str.strip()).all()

# 8. Outliers (flag, don't drop) using the IQR rule
for col in ["Acquisition_Cost", "Clicks","Impressions", "Conversion %", "ROI %"]:
    q1, q3 = df[col].quantile([.25, .75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f"{col}: {n_out} outliers")

# 9. Cross-check against the raw file (cleaning didn't lose or change data)
raw = pd.read_csv("marketing_campaign_dataset_realistic_raw.csv")
assert len(raw) == len(df)
assert (raw["Campaign_ID"].values == df["Campaign_ID"].values).all()

Acquisition_Cost: 17043 outliers
Clicks: 13435 outliers
Impressions: 13549 outliers
Conversion %: 819 outliers
ROI %: 11683 outliers


In [64]:
# 8. Outliers (flag, don't drop) using the IQR rule

outlier_cols = [
    "Acquisition_Cost",
    "Clicks",
    "Impressions",
    "Conversion %",
    "ROI %"
]

for col in outlier_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    df[f"{col}_Outlier"] = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    )

    n_out = df[f"{col}_Outlier"].sum()

    print(f"{col}: {n_out} outliers")

Acquisition_Cost: 17043 outliers
Clicks: 13435 outliers
Impressions: 13549 outliers
Conversion %: 819 outliers
ROI %: 11683 outliers


In [65]:
# View records with at least one outlier
outlier_records = df[
    df[[f"{col}_Outlier" for col in outlier_cols]].any(axis=1)
]

outlier_records.head(10)

,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion %,Acquisition_Cost,ROI %,Location,Clicks,Impressions,Engagement_Score,Customer_Segment,Date,Acquisition_Cost_Outlier,Clicks_Outlier,Impressions_Outlier,Conversion %_Outlier,ROI %_Outlier
5,6,NexGen Systems,Search,Women 25-34,30 days,Website,6.94,9086,489.9,New York,6485,146316,6,Outdoor Adventurers,2025-02-17,False,False,False,False,True
13,14,DataTech Solutions,Email,Men 25-34,45 days,Website,7.57,8126,366.5,Houston,5335,128883,6,Fashionistas,2025-09-10,False,False,False,False,True
28,29,Innovate Industries,Email,Women 35-44,15 days,Email,4.96,26547,347.3,New York,9046,277108,5,Tech Enthusiasts,2025-09-29,False,False,False,False,True
36,37,Alpha Innovations,Display,All Ages,60 days,Facebook,0.95,17220,-33.1,Miami,9862,1596229,7,Outdoor Adventurers,2025-04-27,False,False,True,False,False
37,38,TechCorp,Email,Men 25-34,45 days,Email,9.67,6172,368.3,Houston,3724,136346,4,Fashionistas,2025-10-30,False,False,False,False,True
40,41,Alpha Innovations,Search,All Ages,60 days,Google Ads,5.99,74810,51.1,New York,20587,530309,5,Fashionistas,2025-12-25,True,False,False,False,False
43,44,DataTech Solutions,Influencer,Men 18-24,60 days,YouTube,0.58,9962,-66.6,Chicago,11358,1969603,4,Health & Wellness,2025-07-24,False,False,True,False,False
45,46,Alpha Innovations,Search,Women 25-34,30 days,Google Ads,13.86,90482,4.6,Los Angeles,18197,285965,6,Foodies,2025-11-15,True,False,False,True,False
49,50,Innovate Industries,Email,Women 25-34,15 days,Google Ads,6.63,90577,-16.8,New York,25462,605497,7,Foodies,2025-10-20,True,True,False,False,False
50,51,DataTech Solutions,Search,Men 18-24,60 days,Google Ads,5.09,91394,-23.9,Houston,30938,610541,4,Health & Wellness,2025-11-15,True,True,False,False,False


**Nature of the Data**

Acquisition_Cost: Some campaigns may have significantly higher advertising budgets.
Clicks: Popular campaigns may naturally receive unusually high numbers of clicks.
Impressions: Some campaigns may reach a much larger audience than others.
Conversion %: Only 0.41% are flagged, suggesting relatively few unusually high conversion rates.
ROI %: Some campaigns may generate exceptionally high returns.

High impressions and clicks can represent successful campaigns.
High acquisition costs may reflect larger advertising budgets.
High ROI may indicate exceptionally successful campaigns.
Removing these observations could distort your overall marketing performance analysis.

**Outlier Detection**

The Interquartile Range (IQR) method was applied to identify potential outliers in five numerical variables: Acquisition Cost, Clicks, Impressions, Conversion Rate, and ROI. Observations falling below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR were flagged as potential outliers.

The results indicated that Acquisition Cost had 17,043 outliers, Clicks had 13,435, Impressions had 13,549, Conversion Rate had 819, and ROI had 11,683. These observations were retained rather than removed because unusually high values may represent genuine differences in marketing campaign performance. Therefore, the IQR method was used for outlier identification and monitoring rather than automatic data removal.

In [50]:
print("Conversion % mean:", df["Conversion %"].mean().round(2))
print("Acquisition_Cost mean:", df["Acquisition_Cost"].mean().round(0))
print("ROI % mean:", df["ROI %"].mean().round(2))
print("Clicks mean:", df["Clicks"].mean().round(2))
print("Impressions mean:", df["Impressions"].mean().round(2))
print("Engagement_Score mean:", df["Engagement_Score"].mean().round(2))

Conversion % mean: 4.0
Acquisition_Cost mean: 20447.0
ROI % mean: 69.87
Clicks mean: 8871.09
Impressions mean: 462951.32
Engagement_Score mean: 5.42


In [51]:
CTR = ((df["Clicks"].sum())/(df["Impressions"].sum())*100)
print(CTR.round(2))


1.92


In [52]:
df["Campaign_ID"].unique()

array([     1,      2,      3, ..., 199998, 199999, 200000],
      shape=(200000,))

In [53]:
df['Campaign_ID'].nunique()

200000

In [54]:
print("Campaign_Type:",df["Campaign_Type"].unique())
print("Channel_Used:",df["Channel_Used"].unique())
print("Target_Audience:",df["Target_Audience"].unique())
print("Location:",df["Location"].unique())
print("Customer_Segment:",df["Customer_Segment"].unique())

Campaign_Type: ['Display' 'Social Media' 'Email' 'Influencer' 'Search']
Channel_Used: ['Google Ads' 'Facebook' 'Email' 'YouTube' 'Instagram' 'Website']
Target_Audience: ['Women 25-34' 'Men 18-24' 'Men 25-34' 'Women 35-44' 'All Ages']
Location: ['Miami' 'New York' 'Los Angeles' 'Chicago' 'Houston']
Customer_Segment: ['Foodies' 'Fashionistas' 'Tech Enthusiasts' 'Outdoor Adventurers'
 'Health & Wellness']


In [55]:
spend = df["Acquisition_Cost"].to_numpy(dtype=float)
clicks = df["Clicks"].to_numpy(dtype=float)
impressions = df["Impressions"].to_numpy(dtype=float)
cvr_pct = df["Conversion %"].to_numpy(dtype=float)
roi_pct = df["ROI %"].to_numpy(dtype=float)
engagement = df["Engagement_Score"].to_numpy(dtype=float)
channel = df["Channel_Used"].to_numpy()

In [56]:
# Derived fields (same logic as the DAX measures)
conversions = clicks * cvr_pct / 100        # Total Conversions
profit = spend * roi_pct / 100              # Total Profit


In [59]:
print(profit)
print(conversions)

[-1762.66  -8936.64   4387.892 ...  1641.365 -1615.152   382.36 ]
[264.816  319.6164 223.947  ... 105.3642  14.0049 181.9473]


In [60]:
# ---- 2. Overall KPIs ----
kpi = {
    "Total campaigns": len(df),
    "Total cost": spend.sum(),
    "Total profit": profit.sum(),
    "ROI % (weighted)": 100 * profit.sum() / spend.sum(),
    "CVR %": 100 * conversions.sum() / clicks.sum(),
    "CTR %": 100 * clicks.sum() / impressions.sum(),
    "CPC": spend.sum() / clicks.sum(),
    "CPA": spend.sum() / conversions.sum(),
    "Revenue per conversion": (spend.sum() + profit.sum()) / conversions.sum(),
    "Loss-making %": 100 * np.mean(roi_pct < 0),
    "Avg engagement": engagement.mean(),
}
for name, value in kpi.items():
    print(f"{name:24s}{value:>20,.2f}")


Total campaigns                   200,000.00
Total cost                  4,089,466,482.00
Total profit                3,045,020,841.13
ROI % (weighted)                       74.46
CVR %                                   4.41
CTR %                                   1.92
CPC                                     2.30
CPA                                    52.22
Revenue per conversion                 91.11
Loss-making %                          40.04
Avg engagement                          5.42


In [61]:
# ---- 3. Compare with the Power BI dashboard values (edit these to match your report) ----
dashboard = {
    "Total campaigns": 200_000,
    "ROI % (weighted)": 74.46,
    "CVR %": 4.41,
    "CTR %": 1.92,
    "CPA": 52.22,
    "Revenue per conversion": 91.11,
    "Loss-making %": 40.04,
    "Avg engagement": 5.42,
}
print("\nDashboard check")
for name, expected in dashboard.items():
    ok = np.isclose(kpi[name], expected, atol=0.005, rtol=0)   # dashboard rounds to 2 decimals
    print(f"  {'PASS' if ok else 'FAIL'}  {name:24s} computed={kpi[name]:,.2f}  dashboard={expected:,.2f}")



Dashboard check
  PASS  Total campaigns          computed=200,000.00  dashboard=200,000.00
  PASS  ROI % (weighted)         computed=74.46  dashboard=74.46
  PASS  CVR %                    computed=4.41  dashboard=4.41
  PASS  CTR %                    computed=1.92  dashboard=1.92
  PASS  CPA                      computed=52.22  dashboard=52.22
  PASS  Revenue per conversion   computed=91.11  dashboard=91.11
  PASS  Loss-making %            computed=40.04  dashboard=40.04
  PASS  Avg engagement           computed=5.42  dashboard=5.42


In [62]:
# ---- 4. Weighted vs simple average (why the weighting matters) ----
print(f"\nROI weighted {kpi['ROI % (weighted)']:.2f}% vs simple average {roi_pct.mean():.2f}%")
print(f"CVR weighted {kpi['CVR %']:.2f}% vs simple average {cvr_pct.mean():.2f}%")


ROI weighted 74.46% vs simple average 69.87%
CVR weighted 4.41% vs simple average 4.00%


In [63]:
# ---- 5. Channel breakdown, spend share and profit share ----
channels, idx = np.unique(channel, return_inverse=True)
ch_spend = np.bincount(idx, weights=spend)
ch_profit = np.bincount(idx, weights=profit)
ch_conv = np.bincount(idx, weights=conversions)
ch_roi = 100 * ch_profit / ch_spend
ch_cpa = ch_spend / ch_conv
ch_spend_share = 100 * ch_spend / ch_spend.sum()
ch_profit_share = 100 * ch_profit / ch_profit.sum()

print(f"\n{'Channel':12s}{'ROI %':>9s}{'CPA':>9s}{'Spend sh':>10s}{'Profit sh':>11s}")
for i in np.argsort(-ch_roi):
    print(f"{channels[i]:12s}{ch_roi[i]:9.2f}{ch_cpa[i]:9.2f}{ch_spend_share[i]:10.1f}{ch_profit_share[i]:11.1f}")


Channel         ROI %      CPA  Spend sh  Profit sh
Email          244.07    26.45       7.8       25.7
Website        191.99    31.19       8.5       22.0
Google Ads      73.03    52.70      58.2       57.1
Facebook         8.04    83.96       8.3        0.9
YouTube        -17.64   110.11       3.0       -0.7
Instagram      -25.77   122.82      14.1       -4.9
